# RAG Pipeline with Advanced Retrieval Evaluation with RAGAS
# Generate Synthetic Data using RAGAS, build RAG Pipeline with  Advanced Retrievals, Evaluate Performance

"""
This notebook evaluates different retrieval strategies for building codes domain:
- Naive Retrieval (baseline)
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (Cohere Rerank)
- Ensemble Retrieval

Pipeline:
1. Connect to Qdrant Cloud vector database
2. Generate synthetic test data with RAGAS
3. Implement all retrieval strategies
4. Implement RAG with all retrival strategies
5. Evaluate performance with RAGAS metrics
6. Compare and analyze results
"""

# ============================================================================
# SECTION 1: DEPENDENCIES AND API CONFIGURATION
# ============================================================================

In [1]:
import os
import getpass
from uuid import uuid4
import copy
import time
import pandas as pd
import numpy as np
from datasets import Dataset  

# Set up API keys for external services
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")
os.environ["QDRANT_API_KEY"] = getpass.getpass("🔐 Enter your Qdrant API Key: ")
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("Enter your Langchain API Key:")


In [2]:
# LangChain imports for vector store and embeddings
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# RAGAS imports for synthetic data generation and evaluation
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset.synthesizers import (
    default_query_distribution, 
    SingleHopSpecificQuerySynthesizer, 
    MultiHopAbstractQuerySynthesizer, 
    MultiHopSpecificQuerySynthesizer
)

# RAGAS metrics - CORRECTED
from ragas.metrics import (
    LLMContextRecall,
    ContextEntityRecall,
    LLMContextPrecisionWithReference,
    NonLLMContextPrecisionWithReference  # ← YOU WERE MISSING THIS
)
from ragas import RunConfig, EvaluationDataset, evaluate

# Advanced retrieval strategy imports
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ContextualCompressionRetriever, EnsembleRetriever
from langchain_cohere import CohereRerank

# RAG pipeline imports - YOU WERE MISSING THESE
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Document loading imports - YOU WERE MISSING THESE  
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

# LangChain tracing imports
from langchain.callbacks.tracers import LangChainTracer

# NLTK setup
import nltk
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

True

# ============================================================================
# SECTION 2: QDRANT CLOUD CONNECTION AND VECTOR STORE SETUP
# ============================================================================

In [3]:
print("Setting up Qdrant Cloud connection...")

# Initialize Qdrant client with cloud credentials
qdrant_client = QdrantClient(
    url="https://c95924f4-831b-407f-be42-8e424740487b.us-east-1-0.aws.cloud.qdrant.io",
    api_key=os.environ["QDRANT_API_KEY"],
)

# Initialize embedding model (same as used during document ingestion)
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Connect to vector store with proper field mapping for content retrieval
vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name="post_midterm_R",  # Collection name from Qdrant Cloud
    embedding=embedding_model,
    content_payload_key="content",  # Maps to your 'content' field in Qdrant
    metadata_payload_key="metadata"  # Optional: for additional metadata
)

print("✅ Connected to Qdrant Cloud vector store")

# Test connection and content retrieval
test_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
results = test_retriever.get_relevant_documents("What are the requirements for GFCI protection?")

print("🔍 Testing vector store connection:")
for doc in results:
    print(f"Content preview: {doc.page_content[:200]}...")
    print(f"Metadata: {doc.metadata}")
    print("-" * 50)

Setting up Qdrant Cloud connection...
✅ Connected to Qdrant Cloud vector store


/tmp/ipykernel_26432/1531599195.py:25: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  results = test_retriever.get_relevant_documents("What are the requirements for GFCI protection?")


🔍 Testing vector store connection:
Content preview: of an existing electrical system shall be required to meet
installation and equipment requirements in NFPA 99.
806.4 Residential occupancies. In Group R-2, R-3 and R-4
occupancies and buildings regula...
Metadata: {'_id': '9678ceb3-ea05-5e68-bc04-91cf260f0bee', '_collection_name': 'post_midterm_R'}
--------------------------------------------------
Content preview:  are intended exclusively to
improve the lateral force-resisting system and are not
required by other sections of this code shall not be required
to meet the requirements of Section 1609 or Section 16...
Metadata: {'_id': '184fa3e9-a880-503d-abd9-c7c76638fac6', '_collection_name': 'post_midterm_R'}
--------------------------------------------------
Content preview: 1.
Cables used for survivability of required critical
circuits shall be listed in accordance with UL 2196
and shall have a fire-resistance rating of not less
than 2 hours.
2.
Electrical circuit protec...
Metadata

# ============================================================================
# SECTION 3: SYNTHETIC DATA GENERATION WITH RAGAS
# ============================================================================

Load PDFs for RAGAS (vs feeding qdrant vector store) because
RAGAS works better with complete, coherent documents
Avoids fragmented chunks that might not make sense
Gives RAGAS more context to generate meaningful questions

In [ ]:
print("Loading PDFs from local directory...")

# Load PDFs from local docs/data folder
from pathlib import Path
from collections import defaultdict
from langchain_core.documents import Document
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
# Make randomness reproducible across runs (affects shuffles/sampling, not the LLM itself)
import os, random
import numpy as np

# Stabilize Python's hash-based operations
os.environ["PYTHONHASHSEED"] = "0"

# Seed Python's RNG and NumPy for deterministic sampling/shuffling
random.seed(42)
np.random.seed(42)

# Load and consolidate PDFs by file
data_dir = Path("../docs/data")
raw_documents = DirectoryLoader(str(data_dir), glob="**/*.pdf", loader_cls=PyMuPDFLoader).load()

# Group pages by file
by_file = defaultdict(list)
for d in raw_documents:
    by_file[d.metadata.get("source")].append(d)

# Consolidate pages into full documents
full_docs = []
for src, pages in by_file.items():
    pages = sorted(pages, key=lambda d: d.metadata.get("page", d.metadata.get("page_number", 0)))
    text = "\n\n".join(p.page_content for p in pages)
    full_docs.append(Document(page_content=text, metadata={"source": src}))

sample_docs = full_docs  # or pick a subset of PDFs if needed

print(f"Loaded {len(raw_documents)} raw pages from PDFs")
print(f"Consolidated into {len(full_docs)} full documents")
print(f"Using {len(sample_docs)} full documents for synthetic data generation")

# Set up RAGAS models with proper wrappers
generator_llm = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-4o-mini",  # More capable model for reasoning tasks
    temperature=0.0,      # 0.0 = most deterministic responses (still subject to external factors)
    request_timeout=120   # Extended timeout for complex operations
))

generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

# Initialize synthetic data generator
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)

# Define query distribution for different question types
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),    # Simple factual questions
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),    # Complex reasoning questions
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),    # Multi-step specific questions
]

# Generate synthetic test dataset
try:
    print(f"Generating test dataset with {len(sample_docs)} documents...")
    dataset = generator.generate_with_langchain_docs(
        sample_docs,
        testset_size=10,
        query_distribution=query_distribution,
    )
    print("Dataset generation completed successfully!")
    print(f"Generated {len(dataset)} test samples")
        
    # Convert to pandas for inspection
    testset_df = dataset.to_pandas()
    print("Columns:", testset_df.columns.tolist())
    
except Exception as e:
    print(f"Error during dataset generation: {e}")
    print("Try reducing document count or testset_size further")
    testset_df = None
    dataset = None

# Display sample generated questions
if dataset and testset_df is not None:
    print("\n📋 Sample generated questions:")
    for idx, row in testset_df.head(3).iterrows():
        print(f"{idx + 1}. {row['user_input']}")
        print(f"   Synthesizer: {row['synthesizer_name']}")
        print("-" * 50)

# Final validation check
if dataset and len(dataset) > 0:
    print(f"✅ Ready for evaluation with {len(dataset)} questions")
else:
    print("❌ No dataset generated - check your PDF loading")

Loading PDFs from local directory...
Loaded 3545 raw pages from PDFs
Consolidated into 4 full documents
Using 4 full documents for synthetic data generation
Generating test dataset with 4 documents...


Applying HeadlinesExtractor:   0%|          | 0/4 [00:00<?, ?it/s]

# ============================================================================
# SECTION 4: RAG PIPELINE AND ADVANCED RETRIEVAL STRATEGIES
# ============================================================================
"""
Build complete RAG (Retrieval-Augmented Generation) pipelines that combine:
1. Advanced retrieval strategies for finding relevant documents
2. Augmented prompting for structured responses  
3. LLM generation for final answers
"""

In [5]:
print("🔧 Setting up RAG pipelines and retrieval strategies...")

# RAG prompt template
RAG_PROMPT = """\
You are a North Carolina home inspection expert who answers questions based on provided context. 
You must only use the provided context, and cannot use your own knowledge. 
If the context doesn't contain the answer, respond with "I don't know".

### Question
{question}

### Context
{context}
"""

# Set up components
CHAT_MODEL = ChatOpenAI(model='gpt-4o-mini')
rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

def format_docs(docs):
    """Format list of documents into single context string"""
    return "\n\n".join([doc.page_content for doc in docs])

# 1. Naive Retriever (baseline)
naive_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

naive_rag_chain = (
    {
        "context": itemgetter("question") | naive_retriever | format_docs,
        "question": itemgetter("question")
    }
    | rag_prompt | CHAT_MODEL | StrOutputParser()
)

print("✅ Naive retriever ready")

# 2. BM25 Retriever
# CHANGED: Added conditional creation to handle cases where BM25 fails
# CHANGED: Better error messaging and graceful fallback to None
all_docs = vectorstore.similarity_search("", k=300)
valid_docs = [doc for doc in all_docs if doc.page_content.strip()]

if valid_docs:
    bm25_retriever = BM25Retriever.from_documents(valid_docs)
    bm25_retriever.k = 10
    
    bm25_rag_chain = (
        {
            "context": itemgetter("question") | bm25_retriever | format_docs,
            "question": itemgetter("question")
        }
        | rag_prompt | CHAT_MODEL | StrOutputParser()
    )
    print(f"✅ BM25 retriever created with {len(valid_docs)} documents")
else:
    print("❌ No valid documents for BM25")
    bm25_retriever = None

# 3. Multi-Query Retriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, 
    llm=ChatOpenAI(model="gpt-4o-mini")
)

multi_query_rag_chain = (
    {
        "context": itemgetter("question") | multi_query_retriever | format_docs,
        "question": itemgetter("question")
    }
    | rag_prompt | CHAT_MODEL | StrOutputParser()
)

print("✅ Multi-Query retriever created")

# 4. Contextual Compression with Cohere
compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=naive_retriever
)

compression_rag_chain = (
    {
        "context": itemgetter("question") | compression_retriever | format_docs,
        "question": itemgetter("question")
    }
    | rag_prompt | CHAT_MODEL | StrOutputParser()
)

print("✅ Contextual Compression retriever created")

# 5. Ensemble Retriever
# CHANGED: Made conditional on BM25 success to prevent crashes
# CHANGED: Added warning message when skipped
if bm25_retriever:
    ensemble_retriever = EnsembleRetriever(
        retrievers=[naive_retriever, bm25_retriever],
        weights=[0.5, 0.5]
    )
    
    ensemble_rag_chain = (
        {
            "context": itemgetter("question") | ensemble_retriever | format_docs,
            "question": itemgetter("question")
        }
        | rag_prompt | CHAT_MODEL | StrOutputParser()
    )
    print("✅ Ensemble retriever created")
else:
    ensemble_retriever = None
    print("⚠️ Ensemble retriever skipped (no BM25)")

# Store retrievers for evaluation
# CHANGED: Dynamic retriever dictionary creation instead of hardcoded
# FIXED: Removed non-existent 'parent_document_retriever' reference
retrievers = {
    "Naive": naive_retriever,
    "MultiQuery": multi_query_retriever,
    "ContextualCompression": compression_retriever,
}

if bm25_retriever:
    retrievers["BM25"] = bm25_retriever
    
if ensemble_retriever:
    retrievers["Ensemble"] = ensemble_retriever

print(f"\n✅ Created {len(retrievers)} retrieval strategies")

# Test RAG pipeline
if dataset:
    test_question = testset_df.iloc[0]['user_input']
    test_answer = naive_rag_chain.invoke({"question": test_question})
    print(f"\n🧪 RAG Test - Q: {test_question}")
    print(f"A: {test_answer}")

🔧 Setting up RAG pipelines and retrieval strategies...
✅ Naive retriever ready
✅ BM25 retriever created with 300 documents
✅ Multi-Query retriever created
✅ Contextual Compression retriever created
✅ Ensemble retriever created

✅ Created 5 retrieval strategies

🧪 RAG Test - Q: Wut are the requirements to get a home inspector license in North Carolina by October 1, 2024?
A: I don't know.


# ============================================================================
# SECTION 5: RAG PIPELINE EVALUATION
# ============================================================================

In [19]:
if dataset is None or testset_df is None:
    print("❌ Cannot evaluate - synthetic data generation failed")
else:
    # Set up tracing
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
    os.environ["LANGCHAIN_PROJECT"] = f"BuildingCodes-RAGEval-{uuid4().hex[0:8]}"

    # Configure evaluation
    custom_run_config = RunConfig(timeout=360)
    evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
    
    # Create RAG chains dictionary for evaluation
    rag_chains = {
        "Naive": naive_rag_chain,
        "MultiQuery": multi_query_rag_chain,
        "ContextualCompression": compression_rag_chain,
    }
    
    if bm25_retriever:
        rag_chains["BM25"] = bm25_rag_chain
        
    if ensemble_retriever:
        rag_chains["Ensemble"] = ensemble_rag_chain

    rag_results = {}

    print("\n🔄 Starting RAG pipeline evaluation...")
    print(f"📊 Evaluating {len(rag_chains)} RAG pipelines")

    # Import required metrics
    from ragas.metrics import LLMContextPrecisionWithReference, ResponseRelevancy, Faithfulness, LLMContextRecall

    # Evaluate each RAG pipeline
    for rag_name, rag_chain in rag_chains.items():
        print(f"\n{'='*50}")
        print(f"Evaluating {rag_name} RAG Pipeline")
        print(f"{'='*50}")

        # Create copy of dataset for this RAG chain
        test_dataset_copy = copy.deepcopy(dataset)

        # Generate responses and contexts for each test question
        for test_row in test_dataset_copy:
            # Rate limiting for Cohere-based RAG chains
            if rag_name in ["ContextualCompression", "Ensemble"]:
                time.sleep(6.1)
            
            try:
                # Get the question
                question = test_row.eval_sample.user_input
                
                # Generate response using RAG chain
                response = rag_chain.invoke({"question": question})
                test_row.eval_sample.response = response
                
                # Get retrieved contexts using the corresponding retriever
                retriever = retrievers[rag_name]
                docs = retriever.invoke(question)
                test_row.eval_sample.retrieved_contexts = [doc.page_content for doc in docs]
                
            except Exception as e:
                print(f"⚠️ Error processing {rag_name}: {e}")
                test_row.eval_sample.response = "Error generating response"
                test_row.eval_sample.retrieved_contexts = []

        # Convert to evaluation dataset
        evaluation_dataset = EvaluationDataset.from_pandas(test_dataset_copy.to_pandas())
        
        # Set up tracing
        tracer = LangChainTracer(project_name=f"{os.environ['LANGCHAIN_PROJECT']}-{rag_name}")

        # Run RAG evaluation with specified metrics
        try:
            result = evaluate(
                dataset=evaluation_dataset,
                metrics=[
                    LLMContextPrecisionWithReference(),  # Context Precision
                    ResponseRelevancy(),                 # Response Relevancy  
                    Faithfulness(),                      # Faithfulness
                    LLMContextRecall()                   # Context Recall
                ],
                llm=evaluator_llm,
                run_config=custom_run_config,
                callbacks=[tracer]
            )
            
            rag_results[rag_name] = result
            print(f"✅ {rag_name} RAG pipeline evaluation completed")
            
            # ADDED: Print inline scores immediately after each evaluation
            try:
                print(f"📊 {rag_name} Scores:")
                result_df = result.to_pandas()
                print(f"   • Context Precision: {result_df['llm_context_precision_with_reference'].mean():.4f}")
                print(f"   • Response Relevancy: {result_df['answer_relevancy'].mean():.4f}")
                print(f"   • Faithfulness: {result_df['faithfulness'].mean():.4f}")
                print(f"   • Context Recall: {result_df['context_recall'].mean():.4f}")
                overall_score = (result_df['llm_context_precision_with_reference'].mean() + 
                               result_df['answer_relevancy'].mean() + 
                               result_df['faithfulness'].mean() + 
                               result_df['context_recall'].mean()) / 4
                print(f"   🎯 Overall Score: {overall_score:.4f}")
            except:
                # Fallback if pandas conversion fails
                try:
                    precision_score = getattr(result, 'llm_context_precision_with_reference', 0)
                    relevancy_score = getattr(result, 'answer_relevancy', 0)
                    faithfulness_score = getattr(result, 'faithfulness', 0)
                    recall_score = getattr(result, 'context_recall', 0)
                    print(f"📊 {rag_name} Scores:")
                    print(f"   • Context Precision: {precision_score:.4f}")
                    print(f"   • Response Relevancy: {relevancy_score:.4f}")
                    print(f"   • Faithfulness: {faithfulness_score:.4f}")
                    print(f"   • Context Recall: {recall_score:.4f}")
                    overall_score = (precision_score + relevancy_score + faithfulness_score + recall_score) / 4
                    print(f"   🎯 Overall Score: {overall_score:.4f}")
                except:
                    print(f"⚠️ Could not extract scores for {rag_name}")
            
        except Exception as e:
            print(f"❌ Error evaluating {rag_name}: {e}")
            continue

    print(f"\n✅ RAG pipeline evaluation complete! Results for {len(rag_results)} RAG chains")


🔄 Starting RAG pipeline evaluation...
📊 Evaluating 5 RAG pipelines

Evaluating Naive RAG Pipeline


Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

✅ Naive RAG pipeline evaluation completed
📊 Naive Scores:
   • Context Precision: 0.4039
   • Response Relevancy: 0.6189
   • Faithfulness: 0.5758
   • Context Recall: 0.4667
   🎯 Overall Score: 0.5163

Evaluating MultiQuery RAG Pipeline


Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

✅ MultiQuery RAG pipeline evaluation completed
📊 MultiQuery Scores:
   • Context Precision: 0.4873
   • Response Relevancy: 0.6209
   • Faithfulness: 0.6281
   • Context Recall: 0.4636
   🎯 Overall Score: 0.5500

Evaluating ContextualCompression RAG Pipeline


...

✅ ContextualCompression RAG pipeline evaluation completed
📊 ContextualCompression Scores:
   • Context Precision: 0.5909
   • Response Relevancy: 0.6193
   • Faithfulness: 0.6136
   • Context Recall: 0.2909
   🎯 Overall Score: 0.5287

Evaluating BM25 RAG Pipeline


Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

✅ BM25 RAG pipeline evaluation completed
📊 BM25 Scores:
   • Context Precision: 0.0000
   • Response Relevancy: 0.0000
   • Faithfulness: 0.0000
   • Context Recall: 0.1818
   🎯 Overall Score: 0.0455

Evaluating Ensemble RAG Pipeline


Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

✅ Ensemble RAG pipeline evaluation completed
📊 Ensemble Scores:
   • Context Precision: 0.1887
   • Response Relevancy: 0.6193
   • Faithfulness: 0.6177
   • Context Recall: 0.5576
   🎯 Overall Score: 0.4958

✅ RAG pipeline evaluation complete! Results for 5 RAG chains


# ============================================================================
# SECTION 6: RAG PIPELINE RESULTS ANALYSIS
# ============================================================================

In [21]:
if rag_results:
    print("\n📊 Analyzing RAG pipeline results...")
    
    comparison_data = []
    metrics = ['Context Precision', 'Response Relevancy', 'Faithfulness', 'Context Recall']
    
    # CHANGED: Updated for specified RAG pipeline metrics
    for rag_name, result in rag_results.items():
        try:
            result_df = result.to_pandas()
            row = {
                'RAG Pipeline': rag_name,
                'Context Precision': result_df['llm_context_precision_with_reference'].mean(),
                'Response Relevancy': result_df['answer_relevancy'].mean(),
                'Faithfulness': result_df['faithfulness'].mean(),
                'Context Recall': result_df['context_recall'].mean()
            }
        except Exception as e:
            print(f"⚠️ Error processing {rag_name} results: {e}")
            # Fallback to direct attribute access
            row = {
                'RAG Pipeline': rag_name,
                'Context Precision': getattr(result, 'llm_context_precision_with_reference', 0),
                'Response Relevancy': getattr(result, 'answer_relevancy', 0),
                'Faithfulness': getattr(result, 'faithfulness', 0),
                'Context Recall': getattr(result, 'context_recall', 0)
            }
        
        comparison_data.append(row)

    # Create results DataFrame
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df['Overall Score'] = comparison_df[metrics].mean(axis=1)
    comparison_df_sorted = comparison_df.sort_values('Overall Score', ascending=False)

    print("\n📋 DETAILED RAG PIPELINE RESULTS:")
    print(comparison_df.round(4).to_string(index=False))

    print("\n🏆 RAG PIPELINE PERFORMANCE RANKING:")
    for idx, row in comparison_df_sorted.iterrows():
        print(f"{idx+1}. {row['RAG Pipeline']}: {row['Overall Score']:.4f}")

    print("\n🔍 BEST PERFORMERS BY METRIC:")
    for metric in metrics:
        best_idx = comparison_df[metric].idxmax()
        best_rag = comparison_df.loc[best_idx, 'RAG Pipeline']
        best_score = comparison_df.loc[best_idx, metric]
        print(f"🏆 {metric}: {best_rag} ({best_score:.4f})")

    # Production recommendations
    top_performer = comparison_df_sorted.iloc[0]['RAG Pipeline']
    top_score = comparison_df_sorted.iloc[0]['Overall Score']
    
    print(f"\n🎯 PRODUCTION RECOMMENDATIONS:")
    print(f"🥇 Primary: {top_performer} (Score: {top_score:.4f})")
    print(f"   • Best overall RAG pipeline performance")
    print(f"   • Recommended for production deployment")
    
    if len(comparison_df_sorted) > 1:
        second_performer = comparison_df_sorted.iloc[1]['RAG Pipeline']
        second_score = comparison_df_sorted.iloc[1]['Overall Score']
        print(f"🥈 Alternative: {second_performer} (Score: {second_score:.4f})")
        print(f"   • Good backup RAG pipeline option")

    print(f"\n💡 KEY INSIGHTS FOR RAG PIPELINES:")
    print(f"✅ Generated {len(dataset)} synthetic questions for RAG evaluation")
    print(f"✅ Tested {len(rag_chains)} complete RAG pipelines")
    print(f"✅ {top_performer} emerged as top RAG performer")
    
    # RAG-specific insights for the 4 specified metrics
    precision_leader = comparison_df.loc[comparison_df['Context Precision'].idxmax(), 'RAG Pipeline']
    relevancy_leader = comparison_df.loc[comparison_df['Response Relevancy'].idxmax(), 'RAG Pipeline']
    faithfulness_leader = comparison_df.loc[comparison_df['Faithfulness'].idxmax(), 'RAG Pipeline']
    recall_leader = comparison_df.loc[comparison_df['Context Recall'].idxmax(), 'RAG Pipeline']
    
    print(f"✅ Best context precision: {precision_leader} (highest quality retrieved contexts)")
    print(f"✅ Best response relevancy: {relevancy_leader} (most relevant answers)")
    print(f"✅ Best faithfulness: {faithfulness_leader} (answers stick to context)")
    print(f"✅ Best context recall: {recall_leader} (retrieves most relevant context)")
    
    # Check for concerning patterns
    low_faithfulness = comparison_df[comparison_df['Faithfulness'] < 0.5]
    if not low_faithfulness.empty:
        print(f"⚠️ Low faithfulness detected in: {', '.join(low_faithfulness['RAG Pipeline'].values)}")
        print("   These pipelines may hallucinate or ignore context")
        
    low_precision = comparison_df[comparison_df['Context Precision'] < 0.5]
    if not low_precision.empty:
        print(f"⚠️ Low context precision detected in: {', '.join(low_precision['RAG Pipeline'].values)}")
        print("   These pipelines may retrieve irrelevant contexts")

    print(f"\n🚀 RAG PIPELINE EVALUATION COMPLETE!")
    print(f"📊 Full results available in comparison_df DataFrame")
    print(f"🎯 Use {top_performer} RAG pipeline for production deployment")
    
else:
    print("❌ No RAG pipeline results to analyze - evaluation failed")

print("\n" + "="*60)
print("✅ ADVANCED RETRIEVAL EVALUATION FINISHED")
print("="*60)


📊 Analyzing RAG pipeline results...

📋 DETAILED RAG PIPELINE RESULTS:
         RAG Pipeline  Context Precision  Response Relevancy  Faithfulness  Context Recall  Overall Score
                Naive             0.4039              0.6189        0.5758          0.4667         0.5163
           MultiQuery             0.4873              0.6209        0.6281          0.4636         0.5500
ContextualCompression             0.5909              0.6193        0.6136          0.2909         0.5287
                 BM25             0.0000              0.0000        0.0000          0.1818         0.0455
             Ensemble             0.1887              0.6193        0.6177          0.5576         0.4958

🏆 RAG PIPELINE PERFORMANCE RANKING:
2. MultiQuery: 0.5500
3. ContextualCompression: 0.5287
1. Naive: 0.5163
5. Ensemble: 0.4958
4. BM25: 0.0455

🔍 BEST PERFORMERS BY METRIC:
🏆 Context Precision: ContextualCompression (0.5909)
🏆 Response Relevancy: MultiQuery (0.6209)
🏆 Faithfulness: MultiQue